# Model B: NFL-Realistic

# libaries

In [ ]:
#cell 1
import pandas as pd
import glob
import os

import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

import xgboost as xgb

from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from collections import Counter
import pickle


# load data

In [ ]:
# cell 2
all_csv_files = (
    glob.glob("../1.Data_Collection_Prep/clean_final_datasets/*.csv"))

In [ ]:
#cell 3
for file in all_csv_files:
    file_name = os.path.splitext(os.path.basename(file))[0]
    globals()[file_name] = pd.read_csv(file)


In [ ]:
#cell 4

all_csv_files

In [ ]:
#cell 5

# call dataframe variables
player_personal_data = player_personal_data
college_performance = college_performance_history
player_master = player_master

# Data Info

In [ ]:
# Cell 6: check data structure and NA patterns  
print("College Performance Info:")
print(college_performance.info())
print("\nNA counts in college performance:")
na_counts = college_performance.isnull().sum()
print(na_counts[na_counts > 0].sort_values(ascending=False))
print(f"\nUnique athletes in college data: {college_performance['athlete_id'].nunique()}")


# prep data

## create target + features

In [ ]:
# Cell 9:
# binary target: 0=Not drafted, 1=Drafted
target_df = player_master[['athlete_id', 'drafted']].copy()

# Check target distribution
print(f"\nBinary target distribution:")
print(target_df['drafted'].value_counts())
print(f"As percentages:")
print(target_df['drafted'].value_counts(normalize=True))
print(f"\nTarget breakdown:")
print(f"• Not drafted (0): {(target_df['drafted'] == 0).sum():,} players")
print(f"• Drafted (1): {(target_df['drafted'] == 1).sum():,} players")

In [ ]:
#cell 10
efficiency_columns = [
    'athlete_id',      
    'year',            
    'passing_pct',     
    'passing_ypa',     
    'rushing_ypc',    
    'receiving_ypr',  
    'kicking_pct',    
    'interceptions_avg', 
    'kick_returns_avg',  
    'punt_returns_avg',  
    'punting_ypp'   
]

print("Efficiency metrics for Model B:")
for col in efficiency_columns:
    if col in college_performance.columns:
        if col not in ['athlete_id', 'year']:
            non_null = college_performance[col].notna().sum()
            total = len(college_performance)
            print(f"{col:20}: {non_null:6} non-null ({non_null/total*100:5.1f}%)")

print("\nPhysical features from player_master:")
for col in ['position', 'height', 'weight']:
    if col in player_master.columns:
        non_null = player_master[col].notna().sum()
        total = len(player_master)
        print(f"{col:20}: {non_null:6} non-null ({non_null/total*100:5.1f}%)")

model_b_efficiency = college_performance[efficiency_columns].copy()

print(f"\nModel B efficiency data shape: {model_b_efficiency.shape}")

In [ ]:
#cell 11: aggregate performance efficiency metrics across all seasons
def aggregate_player_efficiency(df):
    numeric_cols = [col for col in df.columns if col not in ['athlete_id', 'year']]
    
    # aggregation functions
    agg_functions = {}
    for col in numeric_cols:
        agg_functions[col] = ['mean', 'max', 'min', 'std', 'count']
    # agg by athlete_id
    player_agg = df.groupby('athlete_id').agg(agg_functions)

    player_agg.columns = [f"{col[0]}_{col[1]}" for col in player_agg.columns]

    player_agg = player_agg.reset_index()
    return player_agg

model_b_efficiency_agg = aggregate_player_efficiency(model_b_efficiency)
print(f"Efficiency features shape: {model_b_efficiency_agg.shape}")

physical_features = player_master[['athlete_id', 'position', 'height', 'weight']].copy()
print(f"Physical features shape: {physical_features.shape}")

print(f"Sample efficiency feature names: {list(model_b_efficiency_agg.columns[1:11])}")
print(f"Physical feature names: {list(physical_features.columns[1:])}")

In [ ]:
# Cell 12: Merge efficiency with binary target variable
model_b_features = model_b_efficiency_agg.merge(
    physical_features, on='athlete_id', how='inner'
)
print(f"After merging efficiency + physical: {model_b_features.shape}")

final_dataset = model_b_features.merge(
    target_df[['athlete_id', 'drafted']], on='athlete_id', how='inner'
)
print(f"Final Model B dataset shape: {final_dataset.shape}")

print("\nEncoding categorical variables...")
final_dataset_encoded = final_dataset.copy()

position_dummies = pd.get_dummies(final_dataset_encoded['position'], prefix='pos')
final_dataset_encoded = pd.concat([final_dataset_encoded, position_dummies], axis=1)

final_dataset_encoded = final_dataset_encoded.drop(['position'], axis=1)

print(f"After encoding categoricals: {final_dataset_encoded.shape}")

# get features and binary target to X and Y
X = final_dataset_encoded.drop(['athlete_id', 'drafted'], axis=1)
y = final_dataset_encoded['drafted']

print(f"\nModel B Feature matrix shape: {X.shape}")
print(f"Features: Efficiency metrics + Height + Weight + Position dummies")
print(f"Binary target distribution:")
print(y.value_counts())
print(f"Class breakdown: Not drafted: {(y==0).sum()}, Drafted: {(y==1).sum()}")

### handle NAs from college history

In [ ]:
# Cell 13: Handle missing values and prepare for modeling

# what percentage of each feature is missing
na_percentages = (X.isnull().sum() / len(X)) * 100
print("Features by percentage of missing values:")
print(na_percentages.sort_values(ascending=False).head(10))

# keep features with <95% missing values
features_to_keep = na_percentages[na_percentages < 95].index.tolist()
print(f"\nKeeping {len(features_to_keep)} features with <95% missing values")

# clean feature matrix
X_clean = X[features_to_keep].copy()
print(f"Clean feature matrix shape: {X_clean.shape}")

# final feature set
print(f"\nFinal Model B features:")
for i, feat in enumerate(X_clean.columns):
    if i % 5 == 0 and i > 0:
        print()  
    print(f"{feat:25}", end=" ")

##  prep data splits

In [ ]:
# Cell 14: Handle missing values and class imbalance

print("Original class distribution:")
print(Counter(y))

# NaN values
print(f"NaN values before imputation: {X_clean.isnull().sum().sum()}")
imputer = SimpleImputer(strategy='constant', fill_value=0)
X_imputed = pd.DataFrame(
    imputer.fit_transform(X_clean),
    columns=X_clean.columns,
    index=X_clean.index
)
print(f"NaN values after imputation: {X_imputed.isnull().sum().sum()}")

# load universal split
with open('test_train_split/athlete_split.pkl', 'rb') as f:
    split = pickle.load(f)

# split based on athlete_id
train_mask = final_dataset['athlete_id'].isin(split['train_athlete_ids'])
test_mask = final_dataset['athlete_id'].isin(split['test_athlete_ids'])

X_train_pre_smote = X_imputed[train_mask]
X_test = X_imputed[test_mask]
y_train_pre_smote = y[train_mask]
y_test = y[test_mask]

# SMOTE only on training data
smote = SMOTE(
    sampling_strategy={
        0: (y_train_pre_smote==0).sum(),  # majority class stays as-is
        1: min((y_train_pre_smote==0).sum() // 3, 15000)   
    },
    random_state=42,
    k_neighbors=5
)

X_train, y_train = smote.fit_resample(X_train_pre_smote, y_train_pre_smote)

print(f"Training: {X_train_pre_smote.shape[0]} → {X_train.shape[0]} (after SMOTE)")
print(f"Test: {X_test.shape[0]} (unchanged)")
print(f"Training class distribution: {Counter(y_train)}")
print(f"Test class distribution: {Counter(y_test)}")

# random forest

## train model

In [ ]:
# Cell 15: train Random Forest
model_a_rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced',
    max_depth=10,
    min_samples_split=50
)

model_a_rf.fit(X_train, y_train)

# predictions
y_pred = model_a_rf.predict(X_test)
y_pred_proba = model_a_rf.predict_proba(X_test)

print("\nRandom Forest Model B (Draft Prediction) Results:")
print(classification_report(y_test, y_pred, target_names=['Not Drafted', 'Drafted']))

# Binary AUC
auc_score = roc_auc_score(y_test, y_pred_proba[:, 1])
print(f"AUC Score: {auc_score:.4f}")

## Assess Model

In [ ]:
# Cell 16: feature importance
import matplotlib.pyplot as plt
import seaborn as sns

feature_importance = pd.DataFrame({
    'feature': X_clean.columns,
    'importance': model_a_rf.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 15 Most Important Efficiency Features in Model B:")
print(feature_importance.head(15))

plt.figure(figsize=(12, 8))
top_features = feature_importance.head(20)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Feature Importance')
plt.title('Model B: Top 20 Most Important Pure Efficiency Features')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Cell 16.5: Analyze Model B feature importance 

# by FEATURE TYPE 
print("\nImportance by feature type:")
feature_types = {
    'Efficiency_Stats': 0,
    'Position_Dummies': 0,
    'Physical_Attributes': 0,
}

for _, row in feature_importance.iterrows():
    feature_name = row['feature']
    importance = row['importance']
    
    if feature_name.startswith('pos_'):
        feature_types['Position_Dummies'] += importance
    elif feature_name in ['height', 'weight']:
        feature_types['Physical_Attributes'] += importance
    else:
        feature_types['Efficiency_Stats'] += importance

for feat_type, importance in sorted(feature_types.items(), key=lambda x: x[1], reverse=True):
    print(f"{feat_type:20}: {importance:.4f}")

print("\nTop efficiency metrics (for Model B comparison):")
efficiency_features = feature_importance[
    ~feature_importance['feature'].str.startswith('pos_') &
    ~feature_importance['feature'].isin(['height', 'weight']) &
    ~feature_importance['feature'].str.contains('team|conference', case=False)
].head(10)

for _, row in efficiency_features.iterrows():
    print(f"{row['feature']:30}: {row['importance']:.4f}")

In [ ]:
# Cell 17: confusion matrix

plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Model B Confusion Matrix (Binary)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label') 
plt.show()

# XGBoost

In [ ]:
# Cell 18: XGBoost (binary classification)
model_a_xgb = xgb.XGBClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',  
    objective='binary:logistic'  
)

model_a_xgb.fit(X_train, y_train)


In [ ]:

# Cell 19: Make predictions
y_pred_xgb = model_a_xgb.predict(X_test)
y_pred_proba_xgb = model_a_xgb.predict_proba(X_test)



In [ ]:
print("\nXGBoost Model B (Draft Prediction) Results:")
print(classification_report(y_test, y_pred_xgb, target_names=['Not Drafted', 'Drafted']))

xgb_auc = roc_auc_score(y_test, y_pred_proba_xgb[:, 1])
print(f"AUC Score: {xgb_auc:.4f}")

In [ ]:
# Cell 20: XGBoost feature importance

xgb_feature_importance = pd.DataFrame({
    'feature': X_clean.columns,
    'importance': model_a_xgb.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 15 Most Important Features - XGBoost Model B:")
print(xgb_feature_importance.head(15))

plt.figure(figsize=(12, 8))
top_features_xgb = xgb_feature_importance.head(20)
plt.barh(range(len(top_features_xgb)), top_features_xgb['importance'])
plt.yticks(range(len(top_features_xgb)), top_features_xgb['feature'])
plt.xlabel('Feature Importance')
plt.title('XGBoost Model B: Top 20 Most Important Pure Efficiency Features')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# cell 20.5
print("\nXGBoost Importance by feature type:")
xgb_feature_types = {
    'Efficiency_Stats': 0,
    'Position_Dummies': 0,
    'Physical_Attributes': 0,
}

for _, row in feature_importance.iterrows():
    feature_name = row['feature']
    importance = row['importance']
    
    if feature_name.startswith('pos_'):
        xgb_feature_types['Position_Dummies'] += importance
    elif feature_name in ['height', 'weight']:
        xgb_feature_types['Physical_Attributes'] += importance
    else:
        xgb_feature_types['Efficiency_Stats'] += importance

for feat_type, importance in sorted(xgb_feature_types.items(), key=lambda x: x[1], reverse=True):
    print(f"{feat_type:20}: {importance:.4f}")

print("\nTop XGBoost efficiency metrics (for Model B comparison):")
xgb_efficiency_features = xgb_feature_importance[
    ~xgb_feature_importance['feature'].str.startswith('pos_') &
    ~xgb_feature_importance['feature'].isin(['height', 'weight']) &
    ~xgb_feature_importance['feature'].str.contains('team|conference', case=False)
].head(10)

for _, row in xgb_efficiency_features.iterrows():
    print(f"{row['feature']:30}: {row['importance']:.4f}")

# Compare

In [ ]:
# Cell 21: Calculate AUC scores 

rf_auc = roc_auc_score(y_test, y_pred_proba[:, 1])
print(f"Random Forest AUC: {rf_auc:.4f}")

xgb_auc = roc_auc_score(y_test, y_pred_proba_xgb[:, 1])
print(f"XGBoost AUC: {xgb_auc:.4f}")

rf_recall_drafted = (y_pred[y_test==1] == 1).mean() if (y_test==1).sum() > 0 else 0
xgb_recall_drafted = (y_pred_xgb[y_test==1] == 1).mean() if (y_test==1).sum() > 0 else 0

print(f"\nRecall for Drafted Players:")
print(f"  Random Forest: {rf_recall_drafted:.1%}")
print(f"  XGBoost:       {xgb_recall_drafted:.1%}")

In [ ]:
# Cell 22: Performance metrics comparison
print(f"ROC AUC Scores:")
print(f"  Random Forest: {rf_auc:.4f}")
print(f"  XGBoost:       {xgb_auc:.4f}")
print(f"  Difference:    {xgb_auc - rf_auc:+.4f}")

print(f"\nRecall for Drafted Players:")
print(f"  Random Forest: {rf_recall_drafted:.1%}")
print(f"  XGBoost:       {xgb_recall_drafted:.1%}")
print(f"  Difference:    {xgb_recall_drafted - rf_recall_drafted:+.1%}")

In [ ]:
##cell 23 Side by side confusion matrices
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# RF confusion matrix
cm_rf = confusion_matrix(y_test, y_pred)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues', ax=ax1)
ax1.set_title('Random Forest Confusion Matrix')
ax1.set_ylabel('True Label')
ax1.set_xlabel('Predicted Label')

# XGBoost confusion matrix  
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Greens', ax=ax2)
ax2.set_title('XGBoost Confusion Matrix')
ax2.set_ylabel('True Label')
ax2.set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 24: Compare feature importance between RF and XGBoost

importance_comparison = feature_importance.merge(
    xgb_feature_importance, 
    on='feature', 
    suffixes=('_rf', '_xgb')
)

# correlation between feature importances
importance_corr = importance_comparison[['importance_rf', 'importance_xgb']].corr().iloc[0,1]
print(f"Feature importance correlation between RF and XGBoost: {importance_corr:.3f}")

# Plot comparison
plt.figure(figsize=(14, 8))
top_20_comparison = importance_comparison.head(20)

x = range(len(top_20_comparison))
width = 0.35

plt.bar([i - width/2 for i in x], top_20_comparison['importance_rf'], 
        width, label='Random Forest', alpha=0.8)
plt.bar([i + width/2 for i in x], top_20_comparison['importance_xgb'], 
        width, label='XGBoost', alpha=0.8)

plt.xlabel('Features')
plt.ylabel('Importance')
plt.title('Model B: Feature Importance Comparison (RF vs XGBoost)')
plt.xticks(x, top_20_comparison['feature'], rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.show()

# biggest differences in feature importance
importance_comparison['diff'] = importance_comparison['importance_xgb'] - importance_comparison['importance_rf']
print("\nFeatures where XGBoost and RF disagree most:")
print("XGBoost values higher:")
print(importance_comparison.nlargest(5, 'diff')[['feature', 'importance_rf', 'importance_xgb', 'diff']])
print("\nRandom Forest values higher:")
print(importance_comparison.nsmallest(5, 'diff')[['feature', 'importance_rf', 'importance_xgb', 'diff']])

In [ ]:
# Cell 25: Final summary 

print(f"Dataset Overview:")
print(f"• Total players: {X_clean.shape[0]:,}")
print(f"• Features: {X_clean.shape[1]} efficiency and physical metrics")
print(f"• Drafted players: {y.sum():,} ({y.mean():.1%})")
print(f"• Class imbalance ratio: {(y==0).sum()//y.sum()}:1")

print(f"\nModel Performance Comparison:")
print(f"{'Metric':<25} {'Random Forest':<15} {'XGBoost':<15} {'Winner'}")
print(f"{'-'*65}")
print(f"{'ROC AUC':<25} {rf_auc:<15.4f} {xgb_auc:<15.4f} {'XGBoost' if xgb_auc > rf_auc else 'RF'}")
print(f"{'Drafted Recall':<25} {rf_recall_drafted:<15.1%} {xgb_recall_drafted:<15.1%} {'XGBoost' if xgb_recall_drafted > rf_recall_drafted else 'RF'}")

print(f"\nTop Performance Drivers (both models agree):")
common_top_features = set(feature_importance.head(10)['feature']) & set(xgb_feature_importance.head(10)['feature'])
for i, feat in enumerate(sorted(common_top_features)[:5], 1):
    print(f"{i}. {feat}")

# save results

In [ ]:

MODEL_SET_NAME = "Model_B"   # change per notebook

results_df = pd.DataFrame({
    "athlete_id": final_dataset.loc[X_test.index, "athlete_id"].values,
    "true_label": y_test.values,

    # Random Forest
    "rf_pred": y_pred,
    "rf_pred_proba": y_pred_proba[:, 1],

    # XGBoost
    "xgb_pred": y_pred_xgb,
    "xgb_pred_proba": y_pred_proba_xgb[:, 1],
})

results_df["model_set"] = MODEL_SET_NAME

os.makedirs("model_results", exist_ok=True)

output_path = f"model_results/{MODEL_SET_NAME}_results.csv"
results_df.to_csv(output_path, index=False)

print(f"saved to: {output_path}")